# Generador automático del PPT mensual — Análisis de Resultados AXI

Este notebook reemplaza las tablas pegadas como imagen en tu PPT por tablas nativas generadas en vivo desde el Excel del mes (hoja **Flash AxI**).

**Cómo usarlo cada mes:**
1. Corré la celda 1 (instala las librerías, una sola vez por sesión de Colab).
2. Corré la celda 2 y subí, cuando te lo pida: el **Excel del mes** y el **PPT de referencia** (el del mes anterior, mismo diseño).
3. En la celda 3, ajustá `MES_ACTUAL` y `MES_COMPARATIVO` (por ejemplo `"sep-26"` y `"sep-25"`).
4. Corré el resto de las celdas en orden.
5. La última celda te descarga el `.pptx` ya armado.


In [ ]:
!pip install -q python-pptx openpyxl
print("Listo.")


In [ ]:
from google.colab import files

print("Subí el Excel del mes (ej. Flash_AXI_09-26.xlsx):")
excel_upload = files.upload()
EXCEL_PATH = list(excel_upload.keys())[0]

print("\nSubí el PPT de referencia (mismo diseño que vas a actualizar):")
pptx_upload = files.upload()
TEMPLATE_PATH = list(pptx_upload.keys())[0]

print("\nArchivos recibidos:", EXCEL_PATH, "/", TEMPLATE_PATH)


In [ ]:
# ---- Ajustá esto cada mes ----
MES_ACTUAL = "ago-26"       # encabezado de la columna del mes actual
MES_COMPARATIVO = "ago-25"  # encabezado de la columna comparativa
OUT_PATH = "Analisis_Res_AXI_autogenerado.pptx"


## Qué tabla va en qué slide

Cada entrada de `TABLE_SPECS` le dice al script qué imagen pegada reemplazar, en qué slide, y de qué filas del Excel sacar los datos. Ya vienen cargadas **Estado de Resultados** (slide 3) y **Venta de servicios** (slide 4).

Para sumar una tabla nueva (Costos operativos, ESF, etc.), copiá uno de los bloques de abajo y cambiá `slide_index`, `picture_name` y `row_range`. Ver la nota al final del notebook para encontrar el `picture_name` de una tabla que todavía no esté acá.

In [ ]:
import openpyxl
from pptx import Presentation
from pptx.util import Emu, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

NAVY = RGBColor(0x0F, 0x2A, 0x4A)
BLUE_HL = RGBColor(0xE8, 0xF1, 0xFC)
WHITE = RGBColor(0xFF, 0xFF, 0xFF)
DARK = RGBColor(0x08, 0x10, 0x18)
GREY_TXT = RGBColor(0x5B, 0x64, 0x72)

TABLE_SPECS = [
    dict(
        slide_index=2,
        picture_name="Imagen 4",
        sheet="Flash AxI",
        row_range=range(10, 33),
        cols=dict(concepto=4, actual=5, comparativo=6, var_abs=7, var_pct=8),
        bold_labels={
            "Total Ventas de servicios", "TOTAL VENTAS", "COSTOS OPERATIVOS",
            "EBITDA", "EBIT", "RESULTADO ANTES DE IIGG", "UTILIDAD (PÉRDIDA) NETA",
        },
        highlight_labels={"EBITDA", "EBIT", "UTILIDAD (PÉRDIDA) NETA"},
        pct_labels={"sobre ventas netas", "s/ ventas netas"},
    ),
    dict(
        slide_index=3,
        picture_name="Imagen 5",
        sheet="Flash AxI",
        row_range=range(10, 18),
        cols=dict(concepto=4, actual=5, comparativo=6, var_abs=7, var_pct=8),
        bold_labels={"Total Ventas de servicios", "TOTAL VENTAS"},
        highlight_labels=set(),
        pct_labels=set(),
    ),
    # Copiá un bloque de estos acá arriba para sumar otra tabla.
]
print(f"{len(TABLE_SPECS)} tablas configuradas.")


In [ ]:
def fmt_num(v):
    if v is None or isinstance(v, str):
        return v or ""
    s = f"{v:,.1f}"
    return s.replace(",", "¤").replace(".", ",").replace("¤", ".")

def fmt_pct(v):
    if v is None or isinstance(v, str):
        return v or ""
    return f"{v * 100:,.1f}".replace(".", ",") + "%"

def extract_rows(ws, spec):
    rows = []
    c = spec["cols"]
    for r in spec["row_range"]:
        concepto = ws.cell(row=r, column=c["concepto"]).value
        if not concepto or not isinstance(concepto, str):
            continue
        label = concepto.strip()
        actual = ws.cell(row=r, column=c["actual"]).value
        comp = ws.cell(row=r, column=c["comparativo"]).value
        var_abs = ws.cell(row=r, column=c["var_abs"]).value
        var_pct = ws.cell(row=r, column=c["var_pct"]).value
        is_pct = label in spec["pct_labels"]
        rows.append(dict(
            label=label,
            actual=fmt_pct(actual) if is_pct else fmt_num(actual),
            comparativo=fmt_pct(comp) if is_pct else fmt_num(comp),
            var_abs="" if is_pct else fmt_num(var_abs),
            var_pct="" if is_pct else fmt_pct(var_pct),
            bold=label in spec["bold_labels"],
            highlight=label in spec["highlight_labels"],
            italic=is_pct,
        ))
    return rows

def set_cell(cell, text, bold=False, align=PP_ALIGN.RIGHT, size=8.5, fill=WHITE, italic=False):
    cell.text = text or ""
    tf = cell.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.alignment = align
    for run in p.runs:
        run.font.size = Pt(size)
        run.font.bold = bold
        run.font.italic = italic
        run.font.name = "Arial"
        run.font.color.rgb = WHITE if fill == NAVY else (NAVY if fill == BLUE_HL else (GREY_TXT if italic else DARK))
    cell.margin_left = Emu(36000)
    cell.margin_right = Emu(36000)
    cell.margin_top = Emu(9000)
    cell.margin_bottom = Emu(9000)
    cell.vertical_anchor = 3
    cell.fill.solid()
    cell.fill.fore_color.rgb = fill

def replace_picture_with_table(slide, spec, rows, mes_actual, mes_comp):
    target = None
    for shp in list(slide.shapes):
        if shp.name == spec["picture_name"]:
            target = shp
            break
    if target is None:
        print(f"  [AVISO] no encontre la imagen '{spec['picture_name']}' en la slide {spec['slide_index'] + 1} - se omite.")
        return
    left, top, width, height = target.left, target.top, target.width, target.height
    target._element.getparent().remove(target._element)

    n_rows = len(rows) + 1
    tbl_shape = slide.shapes.add_table(n_rows, 5, left, top, width, height)
    table = tbl_shape.table

    headers = ["", mes_actual, mes_comp, "Var. $", "Var. %"]
    widths = [Emu(int(width * w)) for w in (0.40, 0.16, 0.16, 0.16, 0.12)]
    for i, w in enumerate(widths):
        table.columns[i].width = w

    for c, h in enumerate(headers):
        set_cell(table.cell(0, c), h, bold=True, size=9, fill=NAVY, align=(PP_ALIGN.LEFT if c == 0 else PP_ALIGN.RIGHT))
    table.rows[0].height = Emu(int(height * 0.07))

    body_h = Emu(int((height - table.rows[0].height) / max(len(rows), 1)))
    for i, row in enumerate(rows, start=1):
        fill = BLUE_HL if row["highlight"] else WHITE
        set_cell(table.cell(i, 0), row["label"], bold=row["bold"], align=PP_ALIGN.LEFT, fill=fill, italic=row["italic"])
        set_cell(table.cell(i, 1), row["actual"], bold=row["bold"], fill=fill, italic=row["italic"])
        set_cell(table.cell(i, 2), row["comparativo"], bold=row["bold"], fill=fill, italic=row["italic"])
        set_cell(table.cell(i, 3), row["var_abs"], bold=row["bold"], fill=fill, italic=row["italic"])
        set_cell(table.cell(i, 4), row["var_pct"], bold=row["bold"], fill=fill, italic=row["italic"])
        table.rows[i].height = body_h

    tbl_el = tbl_shape.table._tbl
    tblPr = tbl_el.find(".//{http://schemas.openxmlformats.org/drawingml/2006/main}tblPr")
    if tblPr is not None:
        tblPr.set("firstRow", "0")
        tblPr.set("bandRow", "0")

print("Funciones cargadas.")


In [ ]:
wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
prs = Presentation(TEMPLATE_PATH)

for spec in TABLE_SPECS:
    print(f"Procesando slide {spec['slide_index'] + 1} ({spec['picture_name']}) desde hoja '{spec['sheet']}'...")
    ws = wb[spec["sheet"]]
    rows = extract_rows(ws, spec)
    slide = prs.slides[spec["slide_index"]]
    replace_picture_with_table(slide, spec, rows, MES_ACTUAL, MES_COMPARATIVO)

prs.save(OUT_PATH)
print(f"\nListo: {OUT_PATH}")


In [ ]:
files.download(OUT_PATH)


---
### Cómo encontrar el `picture_name` de una tabla nueva para automatizar

1. Descomprimí el `.pptx` (es un zip): `!unzip -o tu_deck.pptx -d unpacked`
2. Abrí `unpacked/ppt/slides/slideN.xml` (buscá el número de slide que te interesa).
3. Buscá las etiquetas `<p:cNvPr id="X" name="Imagen Y">` que estén dentro de un `<p:pic>` (no de un `<p:sp>` — esas son formas, no imágenes).
4. Ese `name="Imagen Y"` es el `picture_name`. La posición y el tamaño se toman solos de esa misma imagen, no hace falta calcularlos a mano.
